## SEÇÃO A.1: EXTRACT E STAGING

**Tarefas:**
* Ler o dataset (atenção a encoding e separador)
* **Exploração inicial:** shape, tipos, nulos, estatísticas
* **Diagnóstico de qualidade:** duplicatas, datas como texto, valores inválidos, categorias inconsistentes
* Salvar o bruto em staging com metadados de ingestão: `_source_file`, `_ingested_at`, `_batch_id`

**Registros:** log de cada etapa com qtd antes/depois.


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import uuid
import os
import time
import shutil

In [3]:
ARQUIVO = "../data/datatran2026.csv"

df = pd.read_csv(
    ARQUIVO,
    sep=";",
    encoding="latin1"
)

print("Dataset carregado com sucesso!")

Dataset carregado com sucesso!


In [4]:
print("SHAPE:")
print(df.shape)

print("\n")
print("TIPOS DAS COLUNAS:")
print(df.dtypes)

print("\n")
print("VALORES NULOS:")
print(df.isnull().sum())

print("\n")
print("ESTATÍSTICAS:")
print(df.describe(include="all"))

SHAPE:
(23475, 30)


TIPOS DAS COLUNAS:
id                        int64
data_inversa                str
dia_semana                  str
horario                     str
uf                          str
br                        int64
km                          str
municipio                   str
causa_acidente              str
tipo_acidente               str
classificacao_acidente      str
fase_dia                    str
sentido_via                 str
condicao_metereologica      str
tipo_pista                  str
tracado_via                 str
uso_solo                    str
pessoas                   int64
mortos                    int64
feridos_leves             int64
feridos_graves            int64
ilesos                    int64
ignorados                 int64
feridos                   int64
veiculos                  int64
latitude                    str
longitude                   str
regional                    str
delegacia                   str
uop                         str


In [5]:
print("DIAGNÓSTICO DE QUALIDADE:")

duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

print("\nTipo da coluna data_inversa:")
print(df["data_inversa"].dtype)

datas_convertidas = pd.to_datetime(
    df["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

datas_invalidas = datas_convertidas.isna().sum()

print(f"Datas inválidas: {datas_invalidas}")

colunas_numericas = [
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos"
]

print("\nValores negativos:")

for coluna in colunas_numericas:
    negativos = (df[coluna] < 0).sum()
    print(f"{coluna}: {negativos}")

# Categorias inconsistentes
colunas_categoricas = [
    "dia_semana",
    "uf",
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo",
    "regional",
    "delegacia",
    "uop"
]

print("\nQuantidade de categorias distintas:")

for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].nunique()}")

DIAGNÓSTICO DE QUALIDADE:
Registros duplicados: 0

Tipo da coluna data_inversa:
str
Datas inválidas: 0

Valores negativos:
pessoas: 0
mortos: 0
feridos_leves: 0
feridos_graves: 0
ilesos: 0
ignorados: 0
feridos: 0
veiculos: 0

Quantidade de categorias distintas:
dia_semana: 7
uf: 27
causa_acidente: 68
tipo_acidente: 17
classificacao_acidente: 3
fase_dia: 4
sentido_via: 3
condicao_metereologica: 9
tipo_pista: 3
tracado_via: 355
uso_solo: 2
regional: 28
delegacia: 154
uop: 395


In [6]:
staging = df.copy()

staging["_source_file"] = "datatran2026.csv"

staging["_ingested_at"] = datetime.now()

staging["_batch_id"] = str(uuid.uuid4())

print(staging.head())

       id data_inversa    dia_semana   horario  uf   br     km  municipio  \
0  742921   2026-01-01  quinta-feira  04:04:00  TO  153    155  ARAGUAINA   
1  742942   2026-01-01  quinta-feira  06:40:00  MG  262  146,1  RIO CASCA   
2  742943   2026-01-01  quinta-feira  06:58:00  SC  101    193    BIGUACU   
3  742947   2026-01-01  quinta-feira  07:05:00  DF   60     23   BRASILIA   
4  742960   2026-01-01  quinta-feira  06:17:00  MT  163   1044     MATUPA   

                             causa_acidente                  tipo_acidente  \
0  Objeto estático sobre o leito carroçável                     Tombamento   
1                         Condutor Dormindo                Colisão frontal   
2  Reação tardia ou ineficiente do condutor  Colisão lateral mesmo sentido   
3  Reação tardia ou ineficiente do condutor               Colisão traseira   
4                    Transitar na contramão                Colisão frontal   

   ... feridos veiculos      latitude     longitude regional delegac

In [7]:
os.makedirs("data", exist_ok=True)

staging_path = "../data/staging_datatran2026.csv"

staging.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo em:")
print(staging_path)

Arquivo salvo em:
../data/staging_datatran2026.csv


In [8]:
# Log da etapa
log = pd.DataFrame({
    "Etapa": [
        "Leitura do Dataset",
        "Exploração Inicial",
        "Diagnóstico de Qualidade",
        "Criação da Staging",
        "Gravação da Staging"
    ],
    "Registros Antes": [
        len(df),
        len(df),
        len(df),
        len(df),
        len(staging)
    ],
    "Registros Depois": [
        len(df),
        len(df),
        len(df),
        len(staging),
        len(staging)
    ],
    "Data/Hora": [
        datetime.now(),
        datetime.now(),
        datetime.now(),
        datetime.now(),
        datetime.now()
    ]
})

print(log)

                      Etapa  Registros Antes  Registros Depois  \
0        Leitura do Dataset            23475             23475   
1        Exploração Inicial            23475             23475   
2  Diagnóstico de Qualidade            23475             23475   
3        Criação da Staging            23475             23475   
4       Gravação da Staging            23475             23475   

                   Data/Hora  
0 2026-06-26 01:03:40.526935  
1 2026-06-26 01:03:40.526940  
2 2026-06-26 01:03:40.526940  
3 2026-06-26 01:03:40.526940  
4 2026-06-26 01:03:40.526941  


## SEÇÃO A.2: TRATAMENTO DE QUALIDADE

Trabalhe sobre o staging, nunca sobre o CSV original.

* Remover duplicatas (definir a chave de unicidade)
* Converter tipos (datas -> `datetime`, números -> `float`/`int`)
* Tratar nulos e valores inválidos (e JUSTIFICAR cada decisão)
* Padronizar texto (`strip`, maiúsculas/minúsculas, categorias)

**Documente:** o que removeu, quanto removeu e por quê.

In [9]:
df_tratado = staging.copy()

print(f"Registros iniciais: {len(df_tratado)}")

Registros iniciais: 23475


In [10]:
# Remover duplicatas
antes = len(df_tratado)

df_tratado = df_tratado.drop_duplicates(subset=["id"])

depois = len(df_tratado)

duplicatas_removidas = antes - depois

print(f"Duplicatas removidas: {duplicatas_removidas}")

Duplicatas removidas: 0


# Conversão de tipos

In [11]:
# Data
df_tratado["data_inversa"] = pd.to_datetime(
    df_tratado["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Horário
df_tratado["horario"] = pd.to_datetime(
    df_tratado["horario"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

# Colunas inteiras
colunas_int = [
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos"
]

for coluna in colunas_int:
    df_tratado[coluna] = pd.to_numeric(
        df_tratado[coluna],
        errors="coerce"
    ).astype("Int64")

# Quilometragem
df_tratado["km"] = pd.to_numeric(
    df_tratado["km"],
    errors="coerce"
)

# Latitude e Longitude
df_tratado["latitude"] = pd.to_numeric(
    df_tratado["latitude"],
    errors="coerce"
)

df_tratado["longitude"] = pd.to_numeric(
    df_tratado["longitude"],
    errors="coerce"
)

print(df_tratado.dtypes)

id                                 int64
data_inversa              datetime64[us]
dia_semana                           str
horario                           object
uf                                   str
br                                 int64
km                               float64
municipio                            str
causa_acidente                       str
tipo_acidente                        str
classificacao_acidente               str
fase_dia                             str
sentido_via                          str
condicao_metereologica               str
tipo_pista                           str
tracado_via                          str
uso_solo                             str
pessoas                            Int64
mortos                             Int64
feridos_leves                      Int64
feridos_graves                     Int64
ilesos                             Int64
ignorados                          Int64
feridos                            Int64
veiculos        

# Tratamento de nulos e valores inválidos

In [12]:
# Quantidade de nulos antes
nulos_antes = df_tratado.isnull().sum()

# Campos categóricos
colunas_texto = [
    "dia_semana",
    "uf",
    "municipio",
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo",
    "regional",
    "delegacia",
    "uop"
]

for coluna in colunas_texto:
    df_tratado[coluna] = df_tratado[coluna].fillna("NÃO INFORMADO")

# Campos numéricos
for coluna in colunas_int:
    df_tratado[coluna] = df_tratado[coluna].fillna(0)

# KM
df_tratado["km"] = df_tratado["km"].fillna(0)

# Coordenadas
df_tratado["latitude"] = df_tratado["latitude"].fillna(0)

df_tratado["longitude"] = df_tratado["longitude"].fillna(0)

# Remover valores negativos nas contagens
for coluna in colunas_int:
    df_tratado.loc[df_tratado[coluna] < 0, coluna] = 0

nulos_depois = df_tratado.isnull().sum()

print("Nulos antes:")
print(nulos_antes)

print("\nNulos depois:")
print(nulos_depois)

Nulos antes:
id                            0
data_inversa                  0
dia_semana                    0
horario                       0
uf                            0
br                            0
km                         8693
municipio                     0
causa_acidente                0
tipo_acidente                 0
classificacao_acidente        1
fase_dia                      0
sentido_via                   0
condicao_metereologica        0
tipo_pista                    0
tracado_via                   0
uso_solo                      0
pessoas                       0
mortos                        0
feridos_leves                 0
feridos_graves                0
ilesos                        0
ignorados                     0
feridos                       0
veiculos                      0
latitude                  23458
longitude                 23455
regional                      2
delegacia                     5
uop                          10
_source_file               

# Padronização de texto

In [13]:
for coluna in colunas_texto:

    df_tratado[coluna] = (
        df_tratado[coluna]
        .astype(str)
        .str.strip()
        .str.upper()
    )

print("Padronização concluída.")

Padronização concluída.


# Documentação das transformações

In [14]:
documentacao = pd.DataFrame({

    "Etapa": [

        "Remoção de duplicatas",

        "Conversão de datas",

        "Conversão de tipos numéricos",

        "Tratamento de valores nulos",

        "Tratamento de valores negativos",

        "Padronização de texto"

    ],

    "Quantidade Removida/Alterada": [

        duplicatas_removidas,

        df_tratado["data_inversa"].isna().sum(),

        "-",

        int(nulos_antes.sum() - nulos_depois.sum()),

        "-",

        len(colunas_texto)

    ],

    "Justificativa": [

        "Mantida apenas uma ocorrência por ID do acidente.",

        "Conversão para datetime para permitir análises temporais.",

        "Conversão para tipos numéricos apropriados.",

        "Campos categóricos receberam 'NÃO INFORMADO' e campos numéricos receberam 0.",

        "Contagens negativas são inválidas e foram substituídas por 0.",

        "Remoção de espaços e padronização para letras maiúsculas."

    ]

})

documentacao

,Etapa,Quantidade Removida/Alterada,Justificativa
0,Remoção de duplicatas,0,Mantida apenas uma ocorrência por ID do acidente.
1,Conversão de datas,0,Conversão para datetime para permitir análises...
2,Conversão de tipos numéricos,-,Conversão para tipos numéricos apropriados.
3,Tratamento de valores nulos,55624,Campos categóricos receberam 'NÃO INFORMADO' e...
4,Tratamento de valores negativos,-,Contagens negativas são inválidas e foram subs...
5,Padronização de texto,15,Remoção de espaços e padronização para letras ...


In [15]:
os.makedirs("../data", exist_ok=True)

caminho_tratado = "../data/staging_tratada_datatran2026.csv"

df_tratado.to_csv(
    caminho_tratado,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("Staging tratada salva com sucesso!")
print(f"Local do arquivo: {caminho_tratado}")
print("=" * 50)

Staging tratada salva com sucesso!
Local do arquivo: ../data/staging_tratada_datatran2026.csv


## SEÇÃO A.3: MODELAGEM DIMENSIONAL

* Mínimo de 4 dimensões + 1 fato
* `dim_data` obrigatória (ano, trimestre, mês, dia, `is_weekend`)
* Toda dimensão recebe Surrogate Key (SK) sequencial
* A fato guarda só SKs + métricas

# Dimensão Data

In [16]:
dim_data = (
    df_tratado[
        ["data_inversa", "dia_semana"]
    ]
    .drop_duplicates()
    .sort_values("data_inversa")
    .reset_index(drop=True)
)

# Renomear a coluna de data
dim_data.rename(
    columns={
        "data_inversa": "data"
    },
    inplace=True
)

# Surrogate Key
dim_data.insert(
    0,
    "sk_data",
    range(1, len(dim_data) + 1)
)

# Atributos da dimensão
dim_data["ano"] = dim_data["data"].dt.year
dim_data["trimestre"] = dim_data["data"].dt.quarter
dim_data["mes"] = dim_data["data"].dt.month
dim_data["dia"] = dim_data["data"].dt.day
dim_data["is_weekend"] = dim_data["data"].dt.dayofweek >= 5

dim_data.head()

,sk_data,data,dia_semana,ano,trimestre,mes,dia,is_weekend
0,1,2026-01-01,QUINTA-FEIRA,2026,1,1,1,False
1,2,2026-01-02,SEXTA-FEIRA,2026,1,1,2,False
2,3,2026-01-03,SÁBADO,2026,1,1,3,True
3,4,2026-01-04,DOMINGO,2026,1,1,4,True
4,5,2026-01-05,SEGUNDA-FEIRA,2026,1,1,5,False


# Dimensão Local

In [17]:
dim_local = df_tratado[
    [
        "uf",
        "municipio",
        "br",
        "km",
        "regional",
        "delegacia",
        "uop"
    ]
].drop_duplicates().reset_index(drop=True)

dim_local.insert(0, "sk_local", range(1, len(dim_local)+1))

dim_local.head()

,sk_local,uf,municipio,br,km,regional,delegacia,uop
0,1,TO,ARAGUAINA,153,155.0,SPRF-TO,DEL02-TO,UOP01-DEL02-TO
1,2,MG,RIO CASCA,262,0.0,SPRF-MG,DEL03-MG,UOP03-DEL03-MG
2,3,SC,BIGUACU,101,193.0,SPRF-SC,DEL01-SC,UOP01-DEL01-SC
3,4,DF,BRASILIA,60,23.0,SPRF-DF,DEL03-DF,UOP01-DEL03-DF
4,5,MT,MATUPA,163,1044.0,SPRF-MT,DEL06-MT,UOP03-DEL06-MT


# Dimensão Acidente

In [18]:
dim_acidente = df_tratado[
    [
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente"
    ]
].drop_duplicates().reset_index(drop=True)

dim_acidente.insert(0, "sk_acidente", range(1, len(dim_acidente)+1))

dim_acidente.head()

,sk_acidente,causa_acidente,tipo_acidente,classificacao_acidente
0,1,OBJETO ESTÁTICO SOBRE O LEITO CARROÇÁVEL,TOMBAMENTO,NÃO INFORMADO
1,2,CONDUTOR DORMINDO,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS
2,3,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,COM VÍTIMAS FERIDAS
3,4,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS
4,5,TRANSITAR NA CONTRAMÃO,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS


# Dimensão Via

In [19]:
dim_via = df_tratado[
    [
        "fase_dia",
        "sentido_via",
        "condicao_metereologica",
        "tipo_pista",
        "tracado_via",
        "uso_solo"
    ]
].drop_duplicates().reset_index(drop=True)

dim_via.insert(0, "sk_via", range(1, len(dim_via)+1))

dim_via.head()

,sk_via,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo
0,1,AMANHECER,CRESCENTE,CÉU CLARO,SIMPLES,ACLIVE;RETA,NÃO
1,2,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,RETA,NÃO
2,3,AMANHECER,CRESCENTE,NUBLADO,DUPLA,CURVA,SIM
3,4,PLENO DIA,DECRESCENTE,CÉU CLARO,DUPLA,ACLIVE;RETA,NÃO
4,5,AMANHECER,DECRESCENTE,SOL,SIMPLES,RETA,NÃO


# Associar as Surrogate Keys

In [20]:
# Data
df_fato = df_tratado.merge(
    dim_data,
    left_on="data_inversa",
    right_on="data",
    how="left"
)

# Local
df_fato = df_fato.merge(
    dim_local,
    on=[
        "uf",
        "municipio",
        "br",
        "km",
        "regional",
        "delegacia",
        "uop"
    ],
    how="left"
)

# Acidente
df_fato = df_fato.merge(
    dim_acidente,
    on=[
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente"
    ],
    how="left"
)

# Via
df_fato = df_fato.merge(
    dim_via,
    on=[
        "fase_dia",
        "sentido_via",
        "condicao_metereologica",
        "tipo_pista",
        "tracado_via",
        "uso_solo"
    ],
    how="left"
)

# Tabela fato

In [21]:
fato_acidentes = df_fato[
    [
        "sk_data",
        "sk_local",
        "sk_acidente",
        "sk_via",

        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "feridos",
        "ilesos",
        "veiculos"
    ]
].copy()

# Chave da fato
fato_acidentes.insert(
    0,
    "id_fato",
    range(1, len(fato_acidentes)+1)
)

fato_acidentes.head()

,id_fato,sk_data,sk_local,sk_acidente,sk_via,pessoas,mortos,feridos_leves,feridos_graves,feridos,ilesos,veiculos
0,1,1,1,1,1,3,1,0,0,0,1,5
1,2,1,2,2,2,6,0,0,4,4,1,3
2,3,1,3,3,3,3,0,1,0,1,1,4
3,4,1,4,4,4,3,0,1,0,1,1,3
4,5,1,5,5,5,3,0,0,1,1,1,5


# Conferência do Modelo

In [22]:

print("Quantidade de registros")

print(f"dim_data: {len(dim_data)}")
print(f"dim_local: {len(dim_local)}")
print(f"dim_acidente: {len(dim_acidente)}")
print(f"dim_via: {len(dim_via)}")
print(f"fato_acidentes: {len(fato_acidentes)}")

Quantidade de registros
dim_data: 120
dim_local: 10848
dim_acidente: 1053
dim_via: 3489
fato_acidentes: 23475


# Salvar as tabelas

In [23]:
os.makedirs("dw", exist_ok=True)

# Salvar as dimensões
dim_data.to_csv(
    "dw/dim_data.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_local.to_csv(
    "dw/dim_local.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_acidente.to_csv(
    "dw/dim_acidente.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_via.to_csv(
    "dw/dim_via.csv",
    index=False,
    encoding="utf-8-sig"
)

# Salvar a tabela fato
fato_acidentes.to_csv(
    "dw/fato_acidentes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("Data Warehouse criado com sucesso!")
print("Arquivos salvos em: trilha A/dw/")
print("=" * 50)

Data Warehouse criado com sucesso!
Arquivos salvos em: trilha A/dw/


## SEÇÃO A.4: TÉCNICA 1 - SCD TIPO 2 (OBRIGATÓRIA)

Escolha uma dimensão cujo atributo muda no tempo (ex.: `dim_cliente` com a coluna "segmento").

Implemente o histórico com estas colunas de controle:
* **sk** $\rightarrow$ surrogate key (muda a cada versão)
* **id_natural** $\rightarrow$ id original do registro
* **dt_inicio** $\rightarrow$ início da validade da versão
* **dt_fim** $\rightarrow$ fim da validade (`NULL` = versão atual)
* **flag_atual** $\rightarrow$ `TRUE`/`FALSE`

**Regra:** quando o atributo muda no Lote 2, feche a versão antiga (`dt_fim` + `flag_atual=FALSE`) e crie uma nova linha.

# Criar a dimensão SCD

In [24]:
dim_local_scd = (
    df_tratado[
        [
            "municipio",
            "regional"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Surrogate Key
dim_local_scd.insert(
    0,
    "sk",
    range(1, len(dim_local_scd) + 1)
)

# Chave natural
dim_local_scd.rename(
    columns={"municipio": "id_natural"},
    inplace=True
)

# Controle SCD Tipo 2
dim_local_scd["dt_inicio"] = pd.Timestamp("2026-01-01")
dim_local_scd["dt_fim"] = pd.NaT
dim_local_scd["flag_atual"] = True

dim_local_scd.head()

,sk,id_natural,regional,dt_inicio,dt_fim,flag_atual
0,1,ARAGUAINA,SPRF-TO,2026-01-01,NaT,True
1,2,RIO CASCA,SPRF-MG,2026-01-01,NaT,True
2,3,BIGUACU,SPRF-SC,2026-01-01,NaT,True
3,4,BRASILIA,SPRF-DF,2026-01-01,NaT,True
4,5,MATUPA,SPRF-MT,2026-01-01,NaT,True


# Simulando o Lote 2
 Pois o dataset não possuí clientes, produtos ou funcionários, que são casos clássicos de SDC tipo 2.

In [25]:
# Simulação de um novo lote

lote2 = dim_local_scd.copy()

# Escolhe um município qualquer
municipio_teste = lote2.loc[0, "id_natural"]

print("Município alterado:", municipio_teste)

# Simula mudança de regional
lote2.loc[
    lote2["id_natural"] == municipio_teste,
    "regional"
] = "NOVA REGIONAL"


Município alterado: ARAGUAINA


# Aplicar o SCD Tipo 2

In [26]:
proxima_sk = dim_local_scd["sk"].max() + 1

# Percorre o novo lote
for _, nova_linha in lote2.iterrows():

    atual = dim_local_scd[
        (dim_local_scd["id_natural"] == nova_linha["id_natural"]) &
        (dim_local_scd["flag_atual"] == True)
    ]

    if len(atual) == 0:
        continue

    indice = atual.index[0]

    # Verifica mudança do atributo monitorado
    if atual.iloc[0]["regional"] != nova_linha["regional"]:

        # Fecha versão antiga
        dim_local_scd.loc[indice, "dt_fim"] = pd.Timestamp(datetime.now().date())

        dim_local_scd.loc[indice, "flag_atual"] = False

        # Cria nova versão
        nova_versao = {
            "sk": proxima_sk,
            "id_natural": nova_linha["id_natural"],
            "regional": nova_linha["regional"],
            "dt_inicio": pd.Timestamp(datetime.now().date()),
            "dt_fim": pd.NaT,
            "flag_atual": True
        }

        dim_local_scd = pd.concat(
            [
                dim_local_scd,
                pd.DataFrame([nova_versao])
            ],
            ignore_index=True
        )

        proxima_sk += 1

In [27]:
# Resultado
dim_local_scd.sort_values(
    ["id_natural", "sk"]
).head(20)

,sk,id_natural,regional,dt_inicio,dt_fim,flag_atual
803,804,ABADIA DE GOIAS,SPRF-GO,2026-01-01,NaT,True
1067,1068,ABADIANIA,SPRF-GO,2026-01-01,NaT,True
1695,1696,ABARE,SPRF-PE,2026-01-01,NaT,True
809,810,ABRE CAMPO,SPRF-MG,2026-01-01,NaT,True
59,60,ABREU E LIMA,SPRF-PE,2026-01-01,NaT,True
893,894,ACAILANDIA,SPRF-MA,2026-01-01,NaT,True
877,878,ACARI,SPRF-RN,2026-01-01,NaT,True
541,542,ACAUA,SPRF-PI,2026-01-01,NaT,True
1057,1058,ACEGUA,SPRF-RS,2026-01-01,NaT,True
1377,1378,ACRELANDIA,SPRF-AC,2026-01-01,NaT,True


In [28]:
os.makedirs("dw", exist_ok=True)

# Salvar a dimensão SCD Tipo 2
dim_local_scd.to_csv(
    "dw/dim_local_scd_tipo2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("Dimensão SCD Tipo 2 salva com sucesso!")
print("Local: trilha A/dw/dim_local_scd_tipo2.csv")
print("=" * 50)

Dimensão SCD Tipo 2 salva com sucesso!
Local: trilha A/dw/dim_local_scd_tipo2.csv


## SEÇÃO A.5: LOAD
### CRIAR E POPULAR O DW

* **Conectar:** `duckdb.connect("dw_projeto.duckdb")` $\rightarrow$ prefira o duckdb
* Escrever o DDL (`CREATE TABLE`) de cada tabela:
    * tipos explícitos
    * `PRIMARY KEY` nas SKs
    * `FOREIGN KEY` na fato apontando para as dimensões
* Carregar dimensões primeiro, fato por último
* **Validar:** `COUNT(*)` por tabela e checar SKs nulas na fato

In [29]:
# Instalar e importar DuckDB

!pip install duckdb


'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [30]:
import os

if os.path.exists("dw_projeto.duckdb"):
    os.remove("dw_projeto.duckdb")

In [31]:
# Criar conexão
import duckdb
con = duckdb.connect("dw_projeto.duckdb")

print("Banco criado com sucesso!")

Banco criado com sucesso!


# Criar as tabelas (DDL)

In [32]:
# Remove tabelas caso já existam
con.execute("DROP TABLE IF EXISTS fato_acidentes")
con.execute("DROP TABLE IF EXISTS dim_data")
con.execute("DROP TABLE IF EXISTS dim_local")
con.execute("DROP TABLE IF EXISTS dim_acidente")
con.execute("DROP TABLE IF EXISTS dim_via")

con.execute("""

CREATE TABLE dim_data(

    sk_data INTEGER PRIMARY KEY,

    data DATE,
    
    dia_semana VARCHAR,

    ano INTEGER,

    trimestre INTEGER,

    mes INTEGER,

    dia INTEGER,

    is_weekend BOOLEAN

)

""")

con.execute("""

CREATE TABLE dim_local(

    sk_local INTEGER PRIMARY KEY,

    uf VARCHAR,

    municipio VARCHAR,

    br VARCHAR,

    km DOUBLE,

    regional VARCHAR,

    delegacia VARCHAR,

    uop VARCHAR

)

""")


con.execute("""

CREATE TABLE dim_acidente(

    sk_acidente INTEGER PRIMARY KEY,

    causa_acidente VARCHAR,

    tipo_acidente VARCHAR,

    classificacao_acidente VARCHAR

)

""")


con.execute("""

CREATE TABLE dim_via(

    sk_via INTEGER PRIMARY KEY,

    fase_dia VARCHAR,

    sentido_via VARCHAR,

    condicao_metereologica VARCHAR,

    tipo_pista VARCHAR,

    tracado_via VARCHAR,

    uso_solo VARCHAR

)

""")


con.execute("""

CREATE TABLE fato_acidentes(

    id_fato INTEGER PRIMARY KEY,

    sk_data INTEGER,

    sk_local INTEGER,

    sk_acidente INTEGER,

    sk_via INTEGER,

    pessoas INTEGER,

    mortos INTEGER,

    feridos_leves INTEGER,

    feridos_graves INTEGER,

    feridos INTEGER,

    ilesos INTEGER,

    veiculos INTEGER,

    FOREIGN KEY(sk_data) REFERENCES dim_data(sk_data),

    FOREIGN KEY(sk_local) REFERENCES dim_local(sk_local),

    FOREIGN KEY(sk_acidente) REFERENCES dim_acidente(sk_acidente),

    FOREIGN KEY(sk_via) REFERENCES dim_via(sk_via)

)

""")

print("DDL executado com sucesso!")

DDL executado com sucesso!


# Carregar as dimensões

In [33]:
# Carregar as dimensões

con.register("tmp_dim_data", dim_data)
con.register("tmp_dim_local", dim_local)
con.register("tmp_dim_acidente", dim_acidente)
con.register("tmp_dim_via", dim_via)

con.execute("INSERT INTO dim_data SELECT * FROM tmp_dim_data")

con.execute("INSERT INTO dim_local SELECT * FROM tmp_dim_local")

con.execute("INSERT INTO dim_acidente SELECT * FROM tmp_dim_acidente")

con.execute("INSERT INTO dim_via SELECT * FROM tmp_dim_via")

print("Dimensões carregadas.")

Dimensões carregadas.


# Carregar a tabela fato

In [34]:
con.register("tmp_fato", fato_acidentes)

con.execute("""

INSERT INTO fato_acidentes

SELECT *

FROM tmp_fato

""")

print("Tabela fato carregada.")

Tabela fato carregada.


# Validação (COUNT)

In [35]:
print("Quantidade de registros por tabela\n")

tabelas = [
    "dim_data",
    "dim_local",
    "dim_acidente",
    "dim_via",
    "fato_acidentes"
]

for tabela in tabelas:

    qtd = con.execute(

        f"SELECT COUNT(*) FROM {tabela}"

    ).fetchone()[0]

    print(f"{tabela}: {qtd}")

Quantidade de registros por tabela

dim_data: 120
dim_local: 10848
dim_acidente: 1053
dim_via: 3489
fato_acidentes: 23475


# Validar SKs nulas

In [36]:
consulta = """

SELECT

SUM(CASE WHEN sk_data IS NULL THEN 1 ELSE 0 END) AS sk_data_nula,

SUM(CASE WHEN sk_local IS NULL THEN 1 ELSE 0 END) AS sk_local_nula,

SUM(CASE WHEN sk_acidente IS NULL THEN 1 ELSE 0 END) AS sk_acidente_nula,

SUM(CASE WHEN sk_via IS NULL THEN 1 ELSE 0 END) AS sk_via_nula

FROM fato_acidentes

"""

resultado = con.execute(consulta).fetchdf()

resultado

,sk_data_nula,sk_local_nula,sk_acidente_nula,sk_via_nula
0,0.0,0.0,0.0,0.0


In [37]:
# Encerrar conexão
con.close()

print("Conexão encerrada.")

Conexão encerrada.


## SEÇÃO A.6: TÉCNICA 2 - ÍNDICES (OBRIGATÓRIA)

Crie índices para acelerar as consultas analíticas:

* (CREATE INDEX idx_fato_data ON fato_principal(sk_data);<br>
* CREATE INDEX idx_fato_cliente ON fato_principal(sk_cliente);)

Entrega obrigatória:

* Rode uma consulta com EXPLAIN antes de criar o índice

* Crie o índice

* Rode com EXPLAIN depois

* Compare e comente o plano de execução / tempo




In [38]:
con = duckdb.connect("dw_projeto.duckdb")

# EXPLAIN antes dos índices
consulta = """

SELECT

    d.ano,
    SUM(f.mortos) AS total_mortos

FROM fato_acidentes f

JOIN dim_data d
ON f.sk_data = d.sk_data

WHERE f.sk_data > 100

GROUP BY d.ano

"""

print("PLANO ANTES DOS ÍNDICES\n")

print(
    con.execute(
        "EXPLAIN " + consulta
    ).fetchdf()
)

PLANO ANTES DOS ÍNDICES

     explain_key                                      explain_value
0  physical_plan  ┌───────────────────────────┐\n│         PROJE...


# Criar os índices

In [39]:
con.execute("""

CREATE INDEX idx_fato_data

ON fato_acidentes(sk_data)

""")

con.execute("""

CREATE INDEX idx_fato_local

ON fato_acidentes(sk_local)

""")

print("Índices criados com sucesso!")

Índices criados com sucesso!


In [40]:
# EXPLAIN depois dos índices
print("PLANO DEPOIS DOS ÍNDICES\n")

print(
    con.execute(
        "EXPLAIN " + consulta
    ).fetchdf()
)

PLANO DEPOIS DOS ÍNDICES

     explain_key                                      explain_value
0  physical_plan  ┌───────────────────────────┐\n│         PROJE...


In [41]:
# Comparação de tempo de execução
inicio = time.perf_counter()

con.execute(consulta).fetchall()

fim = time.perf_counter()

print(f"Tempo da consulta: {fim - inicio:.6f} segundos")

Tempo da consulta: 0.015161 segundos


In [42]:
# Encerrar conexão

con.close()

## SEÇÃO A.7: TÉCNICA 3 - PARTICIONAMENTO (OBRIGATÓRIA)

Particionem a fato por uma chave temporal (ex.: ano).

Duas formas aceitas (escolham e justifiquem):
* Exportar a fato em Parquet particionado por ano:

   *  COPY fato_principal TO 'dw_export' (FORMAT PARQUET, PARTITION_BY (ano));

OU manter tabelas separadas por período e comparar.

Entrega: mostrar a estrutura de pastas/partições geradas e explicar como o particionamento reduz a
leitura de dados.

In [43]:
# Conectar ao DuckDB
con = duckdb.connect("dw_projeto.duckdb")

# Criar uma visão da fato com o ano

In [44]:
con.execute("""

CREATE OR REPLACE VIEW vw_fato_particionada AS

SELECT

    f.*,

    d.ano

FROM fato_acidentes f

JOIN dim_data d

ON f.sk_data = d.sk_data

""")

print("View criada com sucesso!")

View criada com sucesso!


# Excluir para evitar sorbescrição de dados

In [45]:
# Remove a pasta caso ela já exista
if os.path.exists("dw_export"):
    shutil.rmtree("dw_export")

# Exportar em Parquet particionado por ano

In [46]:
con.execute("""

COPY vw_fato_particionada

TO 'dw_export'

(
    FORMAT PARQUET,

    PARTITION_BY (ano)

)

""")

print("Exportação concluída.")

Exportação concluída.


# Mostrar a estrutura das partições

In [47]:
for raiz, diretorios, arquivos in os.walk("dw_export"):

    nivel = raiz.replace("dw_export", "").count(os.sep)

    espacos = "    " * nivel

    print(f"{espacos}{os.path.basename(raiz)}/")

    for arquivo in arquivos:
        print(f"{espacos}    {arquivo}")

dw_export/
    ano=2026/
        data_0.parquet


# Ler apenas uma partição


In [48]:
resultado = con.execute("""

SELECT *

FROM read_parquet('dw_export/ano=2026/*.parquet')

LIMIT 5

""").fetchdf()

resultado

,id_fato,sk_data,sk_local,sk_acidente,sk_via,pessoas,mortos,feridos_leves,feridos_graves,feridos,ilesos,veiculos,ano
0,1,1,1,1,1,3,1,0,0,0,1,5,2026
1,2,1,2,2,2,6,0,0,4,4,1,3,2026
2,3,1,3,3,3,3,0,1,0,1,1,4,2026
3,4,1,4,4,4,3,0,1,0,1,1,3,2026
4,5,1,5,5,5,3,0,0,1,1,1,5,2026


In [49]:
con.close()

print("Conexão encerrada.")

Conexão encerrada.



## SEÇÃO A.8: ANALYTICS NO DW

Respondam as 5 perguntas com SQL no DW. Cada consulta deve usar `JOIN` com pelo menos 1 dimensão.

Pelo menos 1 consulta deve usar a dimensão SCD Tipo 2 para mostrar um resultando "histórico" vs "atual".

Pelo menos 1 consulta deve usar uma Window Function (`RANK`, `LAG`, `SUM OVER`, `ROW_NUMBER`...)




In [50]:
con = duckdb.connect("dw_projeto.duckdb")

# Pergunta 1
Quais estados registraram mais acidentes, mortos e feridos?

In [51]:
consulta1 = """

SELECT

    l.uf,

    COUNT(*) AS total_acidentes,

    SUM(f.mortos) AS total_mortos,

    SUM(f.feridos) AS total_feridos

FROM fato_acidentes f

JOIN dim_local l

ON f.sk_local = l.sk_local

GROUP BY l.uf

ORDER BY total_acidentes DESC;

"""

con.execute(consulta1).fetchdf()

,uf,total_acidentes,total_mortos,total_feridos
0,MG,2985,269.0,3816.0
1,SC,2847,138.0,3341.0
2,PR,2561,192.0,2932.0
3,RJ,2099,122.0,2493.0
4,RS,1488,108.0,1703.0
5,SP,1416,57.0,1572.0
6,BA,1344,177.0,1798.0
7,GO,1101,107.0,1225.0
8,PE,1086,111.0,1222.0
9,MT,847,74.0,873.0


# Pergunta 2
Como os acidentes evoluem ao longo do tempo ?

In [52]:
consulta2 = """

SELECT

    d.ano,

    d.mes,

    COUNT(*) AS total_acidentes,

    LAG(COUNT(*)) OVER(

        ORDER BY d.ano, d.mes

    ) AS acidentes_mes_anterior

FROM fato_acidentes f

JOIN dim_data d

ON f.sk_data = d.sk_data

GROUP BY

    d.ano,

    d.mes

ORDER BY

    d.ano,

    d.mes;

"""

con.execute(consulta2).fetchdf()

,ano,mes,total_acidentes,acidentes_mes_anterior
0,2026,1,5822,<NA>
1,2026,2,5591,5822
2,2026,3,6119,5591
3,2026,4,5943,6119


# Pergunta 3
Quais são as principais causas dos acidentes ?

In [53]:
consulta3 = """

SELECT

    a.causa_acidente,

    COUNT(*) AS total_acidentes,

    SUM(f.mortos) AS mortos,

    SUM(f.feridos) AS feridos

FROM fato_acidentes f

JOIN dim_acidente a

ON f.sk_acidente = a.sk_acidente

GROUP BY a.causa_acidente

ORDER BY total_acidentes DESC;

"""

con.execute(consulta3).fetchdf()

,causa_acidente,total_acidentes,mortos,feridos
0,AUSÊNCIA DE REAÇÃO DO CONDUTOR,3701,247.0,4170.0
1,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,3476,181.0,4021.0
2,ACESSAR A VIA SEM OBSERVAR A PRESENÇA DOS OUTR...,2270,133.0,2915.0
3,CONDUTOR DEIXOU DE MANTER DISTÂNCIA DO VEÍCULO...,1387,38.0,1594.0
4,VELOCIDADE INCOMPATÍVEL,1310,155.0,1712.0
...,...,...,...,...
63,FARÓIS DESREGULADOS,2,1.0,1.0
64,DEIXAR DE ACIONAR O FAROL DA MOTOCICLETA (OU S...,2,0.0,4.0
65,SINALIZAÇÃO ENCOBERTA,2,0.0,4.0
66,MODIFICAÇÃO PROIBIDA,1,0.0,2.0


# Pergunta 4
Quais municípios concentram mais acidentes ?

In [54]:
consulta4 = """

SELECT

    l.municipio,

    l.uf,

    COUNT(*) AS total_acidentes

FROM fato_acidentes f

JOIN dim_local l

ON f.sk_local = l.sk_local

GROUP BY

    l.municipio,

    l.uf

ORDER BY total_acidentes DESC

LIMIT 20;

"""

con.execute(consulta4).fetchdf()

,municipio,uf,total_acidentes
0,BRASILIA,DF,327
1,DUQUE DE CAXIAS,RJ,266
2,SAO JOSE,SC,256
3,RECIFE,PE,245
4,BETIM,MG,232
5,GUARULHOS,SP,210
6,JOAO PESSOA,PB,193
7,CURITIBA,PR,191
8,PALHOCA,SC,188
9,SAO JOSE DOS PINHAIS,PR,160


# Pergunta 5
Em quais dias da semana ocorrem mais acidentes ?

In [55]:
consulta5 = """

SELECT
    d.dia_semana,
    COUNT(*) AS total_acidentes,
    SUM(f.mortos) AS mortos,
    SUM(f.feridos) AS feridos
FROM fato_acidentes f
JOIN dim_data d
ON f.sk_data = d.sk_data
GROUP BY d.dia_semana
ORDER BY total_acidentes DESC;

"""

con.execute(consulta5).fetchdf()

,dia_semana,total_acidentes,mortos,feridos
0,DOMINGO,3657,342.0,4327.0
1,SÁBADO,3605,318.0,4261.0
2,SEXTA-FEIRA,3578,280.0,4218.0
3,QUINTA-FEIRA,3396,247.0,3938.0
4,SEGUNDA-FEIRA,3168,265.0,3710.0
5,QUARTA-FEIRA,3093,229.0,3529.0
6,TERÇA-FEIRA,2978,237.0,3574.0
